<a href="https://colab.research.google.com/github/All451/auditoria_controle_estoque/blob/main/Conhe%C3%A7a_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import re
import os
import csv
from pathlib import Path

def detectar_delimitador(arquivo, amostra_linhas=10):
    """
    Detecta automaticamente o delimitador do CSV
    """
    delimitadores = ['\t', ';', ',', '|']

    with open(arquivo, 'r', encoding='utf-8') as f:
        # Ler as primeiras linhas para análise
        linhas = [f.readline() for _ in range(amostra_linhas)]

    contadores = {delim: 0 for delim in delimitadores}

    for linha in linhas:
        for delim in delimitadores:
            contadores[delim] += linha.count(delim)

    # Retorna o delimitador com maior contagem
    return max(contadores.items(), key=lambda x: x[1])[0]

def normalizar_documento(documento: str) -> str:
    """
    Normaliza CNPJ ou CPF, removendo caracteres não numéricos
    """
    if pd.isna(documento):
        return ""

    documento_limpo = re.sub(r"\D", "", str(documento))

    if str(documento).startswith('***'):
        documento_limpo = documento_limpo.replace('*', '')

    return documento_limpo

def processar_arquivo_csv(arquivo_entrada: str, pasta_saida: str):
    """
    Processa um único arquivo CSV, remove duplicatas e salva versão limpa
    """
    try:
        print(f"📖 Processando: {os.path.basename(arquivo_entrada)}")

        # Detectar delimitador automaticamente
        delimitador = detectar_delimitador(arquivo_entrada)
        print(f"🔍 Delimitador detectado: '{delimitador}' (repr: {repr(delimitador)})")

        # Ler CSV com o delimitador detectado
        try:
            df = pd.read_csv(arquivo_entrada, encoding='utf-8', delimiter=delimitador, header=None, engine='python')
        except UnicodeDecodeError:
            # Tentar latin-1 se utf-8 falhar
            df = pd.read_csv(arquivo_entrada, encoding='latin-1', delimiter=delimitador, header=None, engine='python')

        print(f"📊 Formato do arquivo: {df.shape[0]} linhas x {df.shape[1]} colunas")

        # Mostrar preview das primeiras linhas para debug
        print("👀 Primeiras linhas do arquivo:")
        for i in range(min(3, len(df))):
            print(f"Linha {i}: {list(df.iloc[i])}")

        # Verificar se tem colunas suficientes
        if df.shape[1] < 2:
            print("❌ Arquivo com formato inválido. Verificando se é outro formato...")

            # Tentar ler como CSV com várias colunas em uma única coluna
            with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                primeira_linha = f.readline().strip()
                print(f"📝 Conteúdo da primeira linha: {primeira_linha}")

            # Tentar split por espaços múltiplos (caso seja formato de texto fixo)
            try:
                df = pd.read_csv(arquivo_entrada, encoding='utf-8', delim_whitespace=True, header=None, engine='python')
                print(f"🔄 Tentando whitespace: {df.shape[1]} colunas")
            except:
                pass

            if df.shape[1] < 2:
                print("❌ Não foi possível processar o formato do arquivo. Pulando...")
                return None

        # Renomear colunas (usar as colunas disponíveis)
        colunas = ['documento', 'nome', 'estado', 'cidade', 'orgao', 'valor']
        for i in range(min(df.shape[1], len(colunas))):
            df = df.rename(columns={i: colunas[i]})

        # Preencher colunas faltantes com valores vazios
        for col in colunas:
            if col not in df.columns:
                df[col] = ""

        # Normalizar documentos
        df['documento_normalizado'] = df['documento'].apply(normalizar_documento)

        # Remover linhas com documento vazio
        df = df[df['documento_normalizado'].str.len() > 0]

        if len(df) == 0:
            print("⚠️  Nenhum documento válido encontrado. Pulando...")
            return None

        # Remover duplicatas
        df_sem_duplicatas = df.drop_duplicates(subset=['documento_normalizado'], keep='first')

        # Criar pasta de saída
        os.makedirs(pasta_saida, exist_ok=True)

        # Nome do arquivo de saída
        nome_arquivo = os.path.basename(arquivo_entrada)
        nome_saida = f"limpo_{nome_arquivo}"
        caminho_saida = os.path.join(pasta_saida, nome_saida)

        # Salvar CSV limpo
        df_sem_duplicatas.to_csv(caminho_saida, index=False, encoding='utf-8', sep=';')

        # Estatísticas
        total_original = len(df)
        total_limpo = len(df_sem_duplicatas)
        duplicatas_removidas = total_original - total_limpo

        print(f"✅ CSV limpo salvo: {nome_saida}")
        print(f"📊 Registros: {total_original} → {total_limpo} (removidas {duplicatas_removidas} duplicatas)")
        print(f"📝 Documentos únicos: {df_sem_duplicatas['documento_normalizado'].nunique()}")

        return {
            'arquivo': nome_arquivo,
            'original': total_original,
            'limpo': total_limpo,
            'duplicatas': duplicatas_removidas
        }

    except Exception as e:
        print(f"❌ Erro ao processar {os.path.basename(arquivo_entrada)}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def processar_pasta_completa(pasta_entrada: str, pasta_saida: str):
    """
    Processa todos os arquivos CSV de uma pasta
    """
    print(f"🚀 Iniciando processamento da pasta: {pasta_entrada}")
    print("-" * 50)

    # Encontrar todos os arquivos CSV na pasta de entrada
    arquivos_csv = []
    for arquivo in os.listdir(pasta_entrada):
        if arquivo.lower().endswith('.csv'):
            arquivos_csv.append(os.path.join(pasta_entrada, arquivo))

    if not arquivos_csv:
        print("❌ Nenhum arquivo CSV encontrado na pasta de entrada")
        return

    print(f"📁 Encontrados {len(arquivos_csv)} arquivos CSV")

    # Processar cada arquivo
    estatisticas = []
    for arquivo in arquivos_csv:
        stats = processar_arquivo_csv(arquivo, pasta_saida)
        if stats:
            estatisticas.append(stats)
        print("-" * 30)

    # Resumo final
    if estatisticas:
        print("🎯 RESUMO FINAL")
        print("-" * 50)

        total_original = sum(s['original'] for s in estatisticas)
        total_limpo = sum(s['limpo'] for s in estatisticas)
        total_duplicatas = sum(s['duplicatas'] for s in estatisticas)

        print(f"📊 Total de arquivos processados: {len(estatisticas)}")
        print(f"📊 Total de registros originais: {total_original}")
        print(f"📊 Total de registros únicos: {total_limpo}")
        print(f"📊 Total de duplicatas removidas: {total_duplicatas}")

        if total_original > 0:
            print(f"📊 Redução: {total_duplicatas/total_original*100:.1f}%")

        # Salvar relatório
        df_relatorio = pd.DataFrame(estatisticas)
        relatorio_path = os.path.join(pasta_saida, "relatorio_processamento.csv")
        df_relatorio.to_csv(relatorio_path, index=False, encoding='utf-8', sep=';')
        print(f"📋 Relatório salvo: {relatorio_path}")

# Configurações
PASTA_ENTRADA = "/content/ENTRADA"
PASTA_SAIDA = "/content/SAIDA"

# Criar pasta de saída se não existir
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Executar processamento
processar_pasta_completa(PASTA_ENTRADA, PASTA_SAIDA)

🚀 Iniciando processamento da pasta: /content/ENTRADA
--------------------------------------------------
📁 Encontrados 1 arquivos CSV
📖 Processando: BANCO DE DADOS FORNECEDORES - Página1.csv
🔍 Delimitador detectado: ',' (repr: ',')
📊 Formato do arquivo: 65535 linhas x 6 colunas
👀 Primeiras linhas do arquivo:
Linha 0: ['18335404000100', 'ALVO CERTO ARMAS & MUNICOES LTDA', 'MG', 'ARCOS', 'COMANDO DA 4A REGIAO MILITAR', 'R$ 8.040,00']
Linha 1: ['4611672000194', 'ABDON DISTRIBUICAO DE SUPRIMENTOS LTDA', 'GO', 'TRINDADE', '58 BATALHAO DE INFANTARIA MOTORIZADO', 'R$ 1.368,50']
Linha 2: ['***.724.409-**', 'PEDRO SOUZA SPECK', 'SC', 'ITAPOA', 'MINISTERIO DO DESENVOLVIMENTO E ASSISTENCIA SOCIAL, FAMILIA', 'R$ 11.990,00']
✅ CSV limpo salvo: limpo_BANCO DE DADOS FORNECEDORES - Página1.csv
📊 Registros: 65535 → 17173 (removidas 48362 duplicatas)
📝 Documentos únicos: 17173
------------------------------
🎯 RESUMO FINAL
--------------------------------------------------
📊 Total de arquivos processados: